<a target="_blank" href="https://colab.research.google.com/github/agensflow-ai/agensflow-langgraph/blob/main/notebooks/quickstart_adapted.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# AgensFlow LangGraph — quickstart (modern-model adapted)

This notebook uses `security_v1_adapted.json` — the paper priors merged with
120 fresh runs on modern models (claude-haiku-4.5, claude-sonnet-5,
thinkingmachines/inkling). Every arm has 25+ real visits; reward means are
calibrated to the specific model tiers listed above.

**Use this notebook if:** you plan to use the same three model tiers the
substrate was adapted on. You get a stronger starting policy (visible in
Section 4 as tighter arm-level differentiation).

For a **model-agnostic** starter with just paper priors, use `quickstart.ipynb`.

**Prerequisites:** `OPENROUTER_API_KEY`. Runtime: ~5-8 min. Cost: ~$1-2.

## 0. Colab setup (auto-skipped if running locally)

One-shot cell: installs the two packages from PyPI, clones the repo to get
the `examples/` directory the graph builder imports from, `cd`s into the
notebooks folder so the relative paths in later cells resolve, and prompts
for `OPENROUTER_API_KEY` if it isn't already in the environment.

In [ ]:
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    subprocess.run(
        ['pip', 'install', '-q',
         'agensflow-mcp', 'agensflow-langgraph',
         'asgi-lifespan', 'langchain-openai', 'python-dotenv'],
        check=True,
    )
    # Clone the repo to get examples/prompts + documents/starter_policies.
    if not os.path.isdir('agensflow-langgraph'):
        subprocess.run(
            ['git', 'clone', '--depth', '1',
             'https://github.com/agensflow-ai/agensflow-langgraph.git'],
            check=True,
        )
    os.chdir('agensflow-langgraph/notebooks')
    print(f'  ✓ Colab environment ready — cwd = {os.getcwd()}')

# --- OpenRouter API key setup ---
# Option 1 (preferred): a `.env` file with:
#     OPENROUTER_API_KEY=your_key_here
# Option 2: in-cell magic:
#     %env OPENROUTER_API_KEY=your_key_here
# Option 3: paste when prompted below.

from dotenv import load_dotenv
load_dotenv()
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')

if not OPENROUTER_API_KEY:
    print('⚠️  OPENROUTER_API_KEY not found. Set it with %env, a .env file, '
          'or paste it below.')
    OPENROUTER_API_KEY = input('OPENROUTER_API_KEY: ').strip()

os.environ['OPENROUTER_API_KEY'] = OPENROUTER_API_KEY
print('✅ OPENROUTER_API_KEY loaded')

## 1. Boot the policy server in-process

We use `httpx.ASGITransport` to run the FastAPI app inside this Python
process — same trick our integration tests use. All `/langgraph/*` requests
go through the ASGI transport instead of real HTTP, but everything else
(bandits, storage, tenant isolation) works exactly like a real deployment.

In [ ]:
import os

# Force SQLite-in-memory for this notebook run so we don't need ./data/
os.environ.setdefault('AGF_DATABASE_URL', 'sqlite+aiosqlite:///:memory:')
os.environ.setdefault('AGF_ENV', 'test')
os.environ.setdefault('AGF_JWT_SECRET', 'notebook-not-for-production')

from httpx import ASGITransport, AsyncClient
from asgi_lifespan import LifespanManager

from agensflow_mcp.app import create_app
from agensflow_mcp.db.session import init_db, get_engine
from agensflow_mcp.db.models import Base

app = create_app()
await init_db()
engine = get_engine()
async with engine.begin() as conn:
    await conn.run_sync(Base.metadata.create_all)

lifespan_mgr = LifespanManager(app)
await lifespan_mgr.__aenter__()
server_client = AsyncClient(transport=ASGITransport(app=app), base_url='http://test')
print('  ✓ policy server booted in-process')

## 2. Issue an anonymous API key

The same endpoint a real deployment exposes — `POST /auth/anonymous`.

In [ ]:
resp = await server_client.post('/auth/anonymous')
api_key = resp.json()['api_key']
print(f'  api_key: {api_key[:20]}...')

## 3. Route the adapter's HTTP calls through our in-process server

Normally `agensflow-langgraph`'s client hits a real HTTPS server. Here we
monkey-patch it to use the in-process ASGI transport — one small class
override, then all the decorator's `/langgraph/*` calls flow through the
notebook's own kernel.

In [ ]:
from agensflow_langgraph import client as agf_client

class _NotebookClient(agf_client.AgensFlowClient):
    async def _a_post_model(self, path, payload, model_cls):
        r = await server_client.post(path, json=payload, headers=self._headers)
        self._raise_for_status(r)
        return model_cls.model_validate(r.json())

    async def _a_get_model(self, path, model_cls, params=None):
        r = await server_client.get(path, params=params, headers=self._headers)
        self._raise_for_status(r)
        return model_cls.model_validate(r.json())

agf_client._CACHE.clear()
agf_client.AgensFlowClient = _NotebookClient
os.environ['AGENSFLOW_SERVER_URL'] = 'http://test'
os.environ['AGENSFLOW_API_KEY'] = api_key
print('  ✓ adapter wired to in-process server')

## 4. Warm-start from adapted policy (`security_v1_adapted.json`)

This starter is the paper priors merged with 120 real runs on modern models —
a stronger starting policy for the SAME model tier bindings this graph uses.
Reward means already reflect what haiku-4.5 / sonnet-5 / inkling produce on
this domain; UCB will spend fewer runs re-exploring before settling.

**Not model-agnostic:** if you swap in different models, the adapted priors
may not match reality. Use `security_v1.json` (via `quickstart.ipynb`) for
the model-agnostic path.

**Isolation guarantee:** `aimport_policy` writes to YOUR tenant scoped by
`api_key` from Section 2. The starter file on disk is read-only.

In [ ]:
from pathlib import Path
from agensflow_langgraph import aimport_policy

# Path resolves whether we're in local notebooks/ or Colab's cloned notebooks/
candidates = [
    Path('..') / 'examples' / 'starter_policies' / 'security_v1_adapted.json',
    Path('/content') / 'agensflow-langgraph' / 'examples' / 'starter_policies' / 'security_v1_adapted.json',
]
policy_path = next((p for p in candidates if p.exists()), None)
if policy_path is None:
    raise FileNotFoundError(
        'security_v1_adapted.json not found. If you\'re a maintainer, generate it via:\n'
        '    python -m examples.security_domain.converge --epochs 6'
    )

result = await aimport_policy(policy_path)
print(f'  ✓ imported {result["signatures_merged"]} signatures / '
      f'{result["actions_merged"]} arms from the paper-trained warm-start')

## 5. Build the security-domain MAS

6-node LangGraph MAS where **every node is decorated with
`@agensflow(pool={...})`**. Each pool declares 2–3 candidate models; the
substrate learns per-node which one wins.

```
  START → planner → memory → solver ─┬─→ critic  ─┐
                                     └─→ verifier ─┴─→ evaluator → END
```

Critic + verifier run in **parallel** — a shape the decorator handles
without special-casing.

### 5a. Import schemas + prompts (from parallel_critic_mas) + security corpus

The schemas and prompts come from `parallel_critic_mas` unchanged — the
security_domain variant only differs in that MEMORY retrieves from a
per-task subset of the security-advisory corpus (12 synthetic CVEs).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from examples.security_domain.prompts import (
    PlannerOutput, MemoryOutput, SolverOutput,
    VerifierOutput, EvaluatorOutput,
    EvidenceItem,
    PLANNER_SYS, MEMORY_SYS, VERIFIER_SYS, EVALUATOR_SYS,
    SOLVER_SYSTEMS,   # dict: {"concise": SYS, "cot": SYS, "evidence": SYS}
    format_planner_input, format_memory_input, format_solver_input,
    format_verifier_input, format_evaluator_input,
)
from examples.security_domain.corpus import CORPUS, get_corpus_subset
from examples.security_domain.tasks import ALL_TASKS
from examples.security_domain.graph import MODEL_BINDING, SKILL_CARDS

def _render_corpus_subset(doc_ids):
    docs = get_corpus_subset(doc_ids) if doc_ids else CORPUS
    return '\n\n'.join(f'[{d.id}]\n{d.text}' for d in docs)

print(f'  ✓ corpus={len(CORPUS)} docs, {len(ALL_TASKS)} paper tasks')
print(f'  ✓ skill cards: {SKILL_CARDS}')
print(f'  ✓ model tier binding: {MODEL_BINDING}')

### 5b. Declare the per-node pools — skill × model factored

This is the paper's action space, ported. **Solver** is a 3×3 matrix of
(skill card × model tier). **Memory** and **verifier** have a `skip` arm the
substrate learned to pick when the stage isn't worth its cost. Every arm
key below has real paper-learned priors from the warm-start in Section 4.

Model tier binding — the arm-key suffix decides the model:
* `haiku` → `anthropic/claude-haiku-4.5`
* `fast`  → `thinkingmachines/inkling`
* `mini`  → `anthropic/claude-sonnet-5`

That leaves the task pool in two families (Anthropic + ThinkingMachines).
The 3-panel judge in Section 8a is xAI + OpenAI + Qwen — fully disjoint.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableLambda

def or_model(model_id, schema=None):
    # default_headers is load-bearing: OpenRouter uses HTTP-Referer + X-Title
    # for app-tier routing. Without them we hit tighter rate limits.
    llm = ChatOpenAI(
        base_url='https://openrouter.ai/api/v1',
        api_key=os.environ['OPENROUTER_API_KEY'],
        model=model_id, temperature=0.0, max_retries=2,
        default_headers={
            'HTTP-Referer': 'https://agensflow.ai',
            'X-Title': 'AgensFlow security_domain (notebook)',
        },
    )
    return llm.with_structured_output(schema, method='function_calling') if schema else llm

def skip_runnable(default):
    """No-op Runnable that returns `default` regardless of input.
    The substrate's 'skip' choice picks this — zero LLM cost."""
    return RunnableLambda(lambda _: default)

pools = {
    'planner':   {'default': or_model(MODEL_BINDING['mini'], PlannerOutput)},
    'memory':    {
        'use':  or_model(MODEL_BINDING['fast'], MemoryOutput),
        'skip': skip_runnable(MemoryOutput(evidence=[])),
    },
    'solver':    {
        f'{skill}-{tier}': or_model(MODEL_BINDING[tier], SolverOutput)
        for skill in SKILL_CARDS for tier in ('haiku', 'fast', 'mini')
    },
    'verifier':  {
        'fast':  or_model(MODEL_BINDING['fast'], VerifierOutput),
        'haiku': or_model(MODEL_BINDING['haiku'], VerifierOutput),
        'skip':  skip_runnable(VerifierOutput(verdict='supported', ungrounded_claims=[])),
    },
    'evaluator': {'default': or_model(MODEL_BINDING['mini'], EvaluatorOutput)},
}
for node, arms in pools.items():
    print(f'  {node:<10} {len(arms):>2} arms: {list(arms)}')

### 5c. Decorate the 6 nodes with `@agensflow`

**This is the whole integration.** Each async node function gets
`@agensflow(pool=pools['name'])` prepended. The decorator:

1. Derives a signature from the node identity + graph context
2. Asks the substrate which arm to use (UCB1 over the pool keys)
3. Injects the chosen `model` into the function body
4. Captures cost + tokens + latency on the returned message
5. POSTs the outcome to `/langgraph/decision/execute` so the bandit updates

Everything after the `@agensflow` line is your ordinary LangGraph node.
The decorator is the only AgensFlow-specific code you write.

In [ ]:
import operator
from typing import Annotated, TypedDict
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

from agensflow_langgraph import agensflow

class SecurityMASState(TypedDict, total=False):
    user_task: str; corpus_doc_ids: list[str]
    goal: str; subproblem: str
    evidence: list[dict]
    draft_answer: str; solver_reasoning: str
    verifier_verdict: str; ungrounded_claims: list[str]
    revision_count: int
    final_answer: str; evaluator_reasoning: str
    messages: Annotated[list, add_messages]
    trace: Annotated[list, operator.add]

def _trace(node, model):
    cfg = getattr(model, 'config', None) or {}
    return {'node': node, 'action': (cfg.get('metadata') or {}).get('agensflow_action', '<fell-open>')}

@agensflow(pool=pools['planner'])
async def planner(state, model, config=None):
    r = await model.ainvoke([('system', PLANNER_SYS),
                             ('human', format_planner_input(state['user_task']))])
    return {'goal': r.goal, 'subproblem': r.subproblem, 'trace': [_trace('planner', model)]}

@agensflow(pool=pools['memory'])
async def memory(state, model, config=None):
    r = await model.ainvoke([('system', MEMORY_SYS.format(corpus=_render_corpus_subset(state.get('corpus_doc_ids', [])))),
                             ('human', format_memory_input(state['subproblem']))])
    return {'evidence': [e.model_dump() for e in r.evidence], 'trace': [_trace('memory', model)]}

@agensflow(pool=pools['solver'])
async def solver(state, model, config=None):
    # Solver arm key encodes both skill card and model tier — split it here
    cfg = getattr(model, 'config', None) or {}
    action = (cfg.get('metadata') or {}).get('agensflow_action', 'concise-haiku')
    skill = action.split('-', 1)[0] if '-' in action else 'concise'
    system = SOLVER_SYSTEMS.get(skill, SOLVER_SYSTEMS['concise'])
    ev = [EvidenceItem(**e) for e in state.get('evidence', [])]
    r = await model.ainvoke([('system', system),
                             ('human', format_solver_input(state['subproblem'], ev))])
    return {'draft_answer': r.draft_answer, 'solver_reasoning': r.reasoning,
            'trace': [_trace('solver', model)]}

@agensflow(pool=pools['verifier'])
async def verifier(state, model, config=None):
    ev = [EvidenceItem(**e) for e in state.get('evidence', [])]
    r = await model.ainvoke([('system', VERIFIER_SYS),
                             ('human', format_verifier_input(state['subproblem'],
                                                             state['draft_answer'], ev))])
    return {'verifier_verdict': r.verdict, 'ungrounded_claims': list(r.ungrounded_claims),
            'trace': [_trace('verifier', model)]}

def verifier_gate(state):
    if state.get('verifier_verdict') == 'unsupported' and state.get('revision_count', 0) < 2:
        return 'bump_revision'
    return 'evaluator'

async def bump_revision(state):
    return {'revision_count': state.get('revision_count', 0) + 1}

@agensflow(pool=pools['evaluator'])
async def evaluator(state, model, config=None):
    r = await model.ainvoke([('system', EVALUATOR_SYS),
                             ('human', format_evaluator_input(state['goal'], state['draft_answer'],
                                                              state.get('verifier_verdict', 'unknown')))])
    return {'final_answer': r.final_answer, 'evaluator_reasoning': r.merged_reasoning,
            'trace': [_trace('evaluator', model)]}

print('  ✓ 5 decorated nodes defined + verifier-gate + bump_revision')

### 5d. Wire the StateGraph with the verifier gate

Linear topology + a conditional edge from `verifier`. If the verifier says
`unsupported` and we've used less than 2 revision budgets, we send state
back through `bump_revision → solver`. Otherwise we proceed to `evaluator`.

In [ ]:
graph = StateGraph(SecurityMASState)
graph.add_node('planner',       planner)
graph.add_node('memory',        memory)
graph.add_node('solver',        solver)
graph.add_node('verifier',      verifier)
graph.add_node('bump_revision', bump_revision)
graph.add_node('evaluator',     evaluator)

graph.add_edge(START,           'planner')
graph.add_edge('planner',       'memory')
graph.add_edge('memory',        'solver')
graph.add_edge('solver',        'verifier')
graph.add_conditional_edges(
    'verifier', verifier_gate,
    {'bump_revision': 'bump_revision', 'evaluator': 'evaluator'},
)
graph.add_edge('bump_revision', 'solver')
graph.add_edge('evaluator',     END)

compiled = graph.compile(checkpointer=InMemorySaver())
print('  ✓ graph compiled')
print(f'  nodes: {list(compiled.get_graph().nodes)}')

## 6. Run one paper security task end-to-end

Pick one task from the paper's 60-task security suite — this one is IN
distribution (the substrate has priors for it from Section 4's import).
Watch it route intelligently on the first call.

In [ ]:
# Pick a paper task that needs multi-doc synthesis (C2 class — solver should
# route toward the higher-tier arms based on paper priors)
paper_task = next(t for t in ALL_TASKS if t.id == 'C2.1')
print(f'  paper task: {paper_task.id} (class {paper_task.scenario_class})')
print(f'  question:   {paper_task.user_task}')
print(f'  corpus:     {paper_task.corpus_doc_ids}\n')

result = await compiled.ainvoke(
    {'user_task': paper_task.user_task,
     'corpus_doc_ids': paper_task.corpus_doc_ids,
     'trace': []},
    config={'configurable': {'thread_id': 'notebook_paper_task_1'}},
)

print('  Final answer:')
print(f'    {result["final_answer"]}\n')
print('  Substrate routed the graph as:')
for step in result['trace']:
    print(f'    {step["node"]:<15} → {step["action"]}')

## 7. Inspect the routing decisions server-side

Every routing decision (one per node × one per graph invocation) is
persisted to the server. Fetch them and see per-node cost, tokens, latency,
and the arm the substrate chose. Status is `executed` — cost/tokens
captured, awaiting a reward (which Section 8 will provide via the judge).

In [ ]:
# The run produced 6 decisions (one per node). Fetch them from the server —
# status will be 'executed' (cost + tokens captured, but no reward yet). 
# Section 8 below will fetch a real judge quality and submit it as the reward.

resp = await server_client.get(
    '/langgraph/decisions?limit=10',
    headers={'Authorization': f'Bearer {api_key}'},
)
for d in resp.json()['decisions']:
    print(f'  {d["signature"]:<10} action={d["action"]:<10} '
          f'tokens=in{d["tokens_input"] or 0}/out{d["tokens_output"] or 0}  '
          f'lat={d["latency_s"]:.1f}s  status={d["status"]}')

## 8. Where the value shows up

So far: your graph runs, the substrate routes each node, and cost + tokens
get captured server-side. That's the *plumbing*. The **value** — for a
product team evaluating this — is the story around it:

1. **How good was the answer?** — score it with the free-tier 3-panel judge
2. **Why did the substrate pick these arms?** — audit one decision end-to-end
3. **Does it actually learn?** — run a few more queries, watch the policy move

### 8a. Score the final answer with the 3-judge panel

The free tier ships a cross-family judge panel: 3 different-family models
score the answer on four axes (*correctness, completeness, precision,
robustness*) and the panel returns the mean.

**Two design constraints matter here:**

1. **Recent-generation judges** — an old judge underscores nuance in a modern
   answer. We use fresh 2026 models on the price/perf frontier.
2. **Disjoint from the task pool** — the task pool uses OpenAI + Anthropic
   (Sonnet-4) + Meta. If the panel included those families, it'd effectively
   be grading its own homework. So the panel is Google + Qwen + xAI — three
   families that don't appear anywhere in the task graph. That's the
   cross-family part of "3-judge cross-family panel."

The panel needs a **baseline** to anchor the scale — the rubric is relative,
not absolute. We generate a cheap single-model baseline first, then compare
the MAS's answer against it.

In [ ]:
from langchain_openai import ChatOpenAI
from agensflow_langgraph.judge_panel import relative_quality

# --- Cheap baseline: single-shot answer, disjoint from BOTH task pool and
#     judge panel so the panel gets a neutral anchor ---
baseline_llm = ChatOpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=os.environ['OPENROUTER_API_KEY'],
    model='thinkingmachines/inkling',   # not in task pool, not in judges
    temperature=0.0,
    default_headers={
        'HTTP-Referer': 'https://agensflow.ai',
        'X-Title': 'AgensFlow security_domain (notebook)',
    },
)
baseline_sys = (
    'Answer concisely using only the provided context. If the context is empty, '
    'answer from general knowledge in one paragraph.'
)
baseline_msg = await baseline_llm.ainvoke([
    ('system', baseline_sys),
    ('human', paper_task.user_task),
])
baseline_answer = baseline_msg.content
print(f'  ✓ baseline generated ({len(baseline_answer)} chars)')

# --- Panel: 3 fresh-2026 judges, disjoint from Anthropic + ThinkingMachines task pool ---
PANEL_MODELS = (
    'x-ai/grok-4.3',
    'openai/gpt-5.4-mini',
    'qwen/qwen3.6-flash',
)
quality, per_axis = await relative_quality(
    task=paper_task.user_task,
    candidate=result['final_answer'],
    baseline=baseline_answer,
    openrouter_key=os.environ['OPENROUTER_API_KEY'],
    models=PANEL_MODELS,
)

print(f'\n  Panel quality (composed):  {quality:.3f}')
print('  Per-axis scores:')
for axis, score in per_axis.items():
    print(f'    {axis:14s} {score:.3f}')
print(f'\n  Panel: {", ".join(PANEL_MODELS)}')

### 8b. Audit one routing decision

Every decision the substrate makes is persisted with the full context. For
any given `decision_id`, you can retrieve: the signature (task-shape hash),
which arm was chosen, cost + tokens + latency, the quality it earned, and the
**current policy state** for that signature (visits + reward_mean per arm).
The last piece is what makes this auditable — you can always answer *"why
was this the reasonable choice given what the substrate knew?"*

In [ ]:
from agensflow_langgraph import arecord_reward
from agensflow_langgraph.client import get_client

# Submit the panel quality from 8a as the real reward.
await arecord_reward(quality=quality, thread_id='notebook_demo_1')
print(f'  ✓ submitted quality={quality:.3f} as reward for thread notebook_demo_1\n')

# Fetch the run's decisions and pick the solver's — the most interesting node
# (3 arms, so the substrate had a real choice to make).
resp = await server_client.get(
    '/langgraph/decisions?limit=20',
    headers={'Authorization': f'Bearer {api_key}'},
)
decisions = resp.json()['decisions']
solver_dec = next(d for d in decisions if d['signature'] == 'solver')

print('  === Decision audit — solver node ===')
print(f'    decision_id:   {solver_dec["decision_id"]}')
print(f'    signature:     {solver_dec["signature"]}')
print(f'    chose arm:     {solver_dec["action"]}')
print(f'    status:        {solver_dec["status"]}')
print(f'    cost:          ${solver_dec["cost_usd"] or 0:.4f}')
print(f'    tokens:        in={solver_dec["tokens_input"] or 0}  out={solver_dec["tokens_output"] or 0}')
print(f'    latency:       {solver_dec["latency_s"] or 0:.2f}s')
if solver_dec['quality'] is not None:
    print(f'    quality:       {solver_dec["quality"]:.3f}')
if solver_dec.get('reward_value') is not None:
    print(f'    reward_value:  {solver_dec["reward_value"]:.4f}')

# What did the substrate KNOW about each solver arm at time of choice?
# aexport_policy returns the WHOLE bandit state; slice the solver signature.
policy_resp = await get_client().a_export_policy()
solver_arms = policy_resp.policy.get('solver', {})

print('\n  === Solver arm stats (post-reward) ===')
print(f'    {"arm":<10} {"visits":>7} {"reward_mean":>12} {"reward_m2":>11}')
for arm, stats in sorted(solver_arms.items()):
    marker = '  ← chosen' if arm == solver_dec['action'] else ''
    print(f'    {arm:<10} {stats.get("visits", 0):>7} {stats.get("reward_mean", 0):>12.4f} '
          f'{stats.get("reward_m2", 0):>11.4f}{marker}')

### 8c. Adapt to novel questions the paper never asked

This is the moment. Run 2 out-of-distribution security questions the paper
never asked — the substrate picks arms based on paper priors, judges the
answer, and updates its beliefs. We snapshot the policy before + after and
print the arm-level delta.

A well-behaved substrate shows: `visits` increment on the arms it picked;
`reward_mean` moves in the direction of the new evidence (up if the paper's
preferred arm keeps winning on OOD tasks, down if it doesn't transfer).

That IS the flywheel — visible in a diff.

In [ ]:
# Snapshot policy BEFORE the OOD runs
policy_before = (await get_client().a_export_policy()).policy
print(f'  policy_before: {sum(len(v) for v in policy_before.values())} arms across '
      f'{len(policy_before)} signatures\n')

# Two novel security questions — none of these exact questions appear in the
# paper's 60-task suite, but they're in the same domain (security advisories).
ood_tasks = [
    ('notebook_ood_1',
     'For which vulnerabilities in the corpus does the fix require '
     'restarting the affected service, and why?',
     [d.id for d in CORPUS]),   # multi-doc synthesis, all 12 docs
    ('notebook_ood_2',
     'Which two advisories share the same disclosure quarter and what class of '
     'vulnerability do they represent?',
     [d.id for d in CORPUS]),
]

for thread_id, task_text, doc_ids in ood_tasks:
    r = await compiled.ainvoke(
        {'user_task': task_text, 'corpus_doc_ids': doc_ids, 'trace': []},
        config={'configurable': {'thread_id': thread_id}},
    )
    # Cheap baseline + 3-panel judge (same pattern as 8a)
    base = (await baseline_llm.ainvoke([('system', 'Answer concisely.'),
                                        ('human', task_text)])).content
    q, _ = await relative_quality(
        task=task_text, candidate=r['final_answer'], baseline=base,
        openrouter_key=os.environ['OPENROUTER_API_KEY'], models=PANEL_MODELS,
    )
    await arecord_reward(quality=q, thread_id=thread_id)
    print(f'  {thread_id}: quality={q:.3f}')
    print(f'    routing: ' + ', '.join(f'{s["node"]}={s["action"]}' for s in r['trace']))

# Snapshot AFTER, diff
policy_after = (await get_client().a_export_policy()).policy

print('\n  === Policy delta after 2 OOD runs ===')
print(f'    {"signature":<12} {"arm":<12} {"visits Δ":>10} {"reward_mean Δ":>15}')
for sig in sorted(set(policy_before) | set(policy_after)):
    b_arms = policy_before.get(sig, {})
    a_arms = policy_after.get(sig, {})
    for arm in sorted(set(b_arms) | set(a_arms)):
        vb = b_arms.get(arm, {}).get('visits', 0)
        va = a_arms.get(arm, {}).get('visits', 0)
        mb = b_arms.get(arm, {}).get('reward_mean', 0.0)
        ma = a_arms.get(arm, {}).get('reward_mean', 0.0)
        dv = va - vb
        dm = ma - mb
        if dv > 0 or abs(dm) > 1e-4:
            arrow = '↑' if dm > 0 else ('↓' if dm < 0 else '·')
            print(f'    {sig:<12} {arm:<12} {dv:>+10} {dm:>+15.4f} {arrow}')

## 9. Cleanup

Close the in-process server.

In [ ]:
await server_client.aclose()
await lifespan_mgr.__aexit__(None, None, None)
print('  ✓ done')

## What you just saw

1. **Section 4** — you imported a bandit policy converged on the paper's
   60-task security-advisory suite. Your fresh tenant inherited months of
   research work in one HTTP POST.
2. **Sections 5-6** — you defined a 6-node LangGraph MAS. The `@agensflow`
   decorator on each node is the ENTIRE integration surface. Every model
   call routed through the substrate; every cost + token was captured.
3. **Section 7** — every decision is persisted with full context: signature,
   chosen arm, cost, tokens, latency. Auditable end-to-end.
4. **Section 8a** — the free-tier 3-panel judge scored the answer on 4 axes
   using 3 cross-family models (Google + Qwen + xAI) that are DISJOINT from
   the task pool (OpenAI + Anthropic + Meta) — no self-scoring bias.
5. **Section 8b** — for any single decision, you can see what the substrate
   knew about each arm at time of choice + how the reward updated after the
   judge scored the answer.
6. **Section 8c** — two novel questions the paper never asked. Substrate
   routed on paper priors, judge scored, bandit updated. The delta table is
   the flywheel — visible in a diff.

**Same substrate, all the way down.** The moat isn't "we route your first
query." It's "we carry forward learned wisdom AND adapt to your specific
case." Enterprise adds the private-tier behavioral observability signals
([MLDX](https://github.com/AgensFlow-ai/agensflow-mcp-enterprise)); the OSS
free tier is everything you just ran.